In [1]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import confusion_matrix


In [12]:
recordings = glob.glob("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0*/recordings.csv")
recordings

[]

In [13]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVW"

In [14]:
def get_all_level(nace_code, df_nace_codes_descriptions= None): 

    if nace_code < 10:
        nace_code = "0" + str(nace_code)

    init_level = len(str(nace_code).replace(".",""))
    
    levels = {init_level: nace_code}

    if df_nace_codes_descriptions is None: 
            df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
    

    try: 
        for i in range(0,init_level-1):     
            parent = df_nace_codes_descriptions[df_nace_codes_descriptions["ID"] == str(df_nace_codes_descriptions[df_nace_codes_descriptions["CODE"] == str(levels[init_level-i])]["PARENT_ID"].iloc[0])]
            levels[init_level-i-1] = parent["CODE"].iloc[0]
    except IndexError:
         return None


    return levels

# Numerics

In [15]:
for recording in sorted(recordings): 
    df = pd.read_csv(recording)
    print("\nNace Level: " + recording.split("nace_level")[1][1])
    for i in range(1,5): 
        if f"position_lvl_{i}" in df.columns: 
            print(f"Mean Position Level {i}: ", round(df[f"position_lvl_{i}"].mean(), 2), " from ", df.loc[0,f"classes_lvl_{i}"], " classes.")


In [16]:
# Initialize a dictionary to store the mean position levels for each recording
mean_positions = {"Recording": [], "Mean Top k% Level 1": [], "Mean Top k% Level 2": [], "Mean Top k% Level 3": [], "Mean Top k% Level 4": []}

# Iterate through each recording and calculate the mean position levels
for recording in sorted(recordings): 
    df = pd.read_csv(recording)
    mean_positions["Recording"].append("Nace Level: " + recording.split("nace_level")[1][1])
    for i in range(1, 5): 
        if f"position_lvl_{i}" in df.columns: 
            mean_positions[f"Mean Top k% Level {i}"].append(round(df[f"position_lvl_{i}"].mean()/df.loc[0,f"classes_lvl_{i}"], 2))
        else:
            mean_positions[f"Mean Top k% Level {i}"].append(None)


# Convert the dictionary to a DataFrame for better visualization
mean_positions_df = pd.DataFrame(mean_positions)
mean_positions_df


,Recording,Mean Top k% Level 1,Mean Top k% Level 2,Mean Top k% Level 3,Mean Top k% Level 4


# Position Distribution

In [17]:
for recording in sorted(recordings): 
    df = pd.read_csv(recording)
    df = df.dropna(subset="position_lvl_1")
    plt.hist(df["position_lvl_1"], alpha = 0.4, density=True)
    plt.xlim([0,21])
    plt.title("Nace Level: " + recording.split("nace_level")[1][1])
    plt.xlabel("Position")
    plt.show()

## Proportion right

In [21]:
csvs = glob.glob("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables/tables_cos_sim_0.0_nace_level_1_stoxx/*/*.txt_long.csv")

In [22]:
dataset_path = "../data/PDF_stoxx600"
nace_classes = pd.read_excel(os.path.join(dataset_path, "STOXX600_as_of_2025_03_13.xlsx"))

In [23]:
dfs = []
for csv in csvs: 
    df = pd.read_csv(csv, index_col=0)
    df.iloc[:, 1:] = df.iloc[:, 1:].apply(lambda row: row.where(row == row.max(), other=pd.NA), axis=1)
    df = pd.melt(df, var_name="NACE_code", value_name="Cos_sim", id_vars=("Sentences"))
    df = df.dropna(subset="Cos_sim")
    
    name = csv.split("/")[-2][:-4] + ".pdf"
    original_code = nace_classes[nace_classes["Report"] == name]["NACE_letter"].iloc[0] 

    df["Original_code"] = original_code
    df["NACE_code_letter"] = df["NACE_code"].apply(lambda x: x.split("_")[1])
    df["Right_classification"] = df["Original_code"] == df["NACE_code_letter"]
    df["Sentence_len"] = df["Sentences"].apply(len)

    dfs.append(df)
full_df = pd.concat(dfs)

In [24]:
full_df["Right_classification"].mean()

0.16413123167155425